In [11]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

# make the package importable
sys.path.insert(0, str(Path.cwd() / "04_src"))

from historyAssistant.planning import route_scenario
from historyAssistant.llm.llm_client import build_openai_client

In [12]:
# mapping ground truth labels
_LABEL_MAP = {
    # v4 scenario tags
    "simple_lookup": "simple_lookup",
    "aggregation": "aggregation",
    "filtering_ranking": "filtering_ranking",
    # baseline category_id tags
    "simple_fact_lookup": "simple_lookup",
    "aggregation_across_time": "aggregation",
    "comparison_delta": "aggregation",
    "trend_seasonality": "aggregation",
    "kpi_sla": "aggregation",
    "distribution_percentile": "aggregation",
    "anomaly_dq": "aggregation",
    "segmentation": "aggregation",
    # baseline query_task_type tags
    "single_value_lookup": "simple_lookup",
    "tabular_result": "simple_lookup",
    "scalar_aggregation": "aggregation",
    "grouped_aggregation": "aggregation",
}

In [13]:
def normalize_expected(record: dict) -> str:
    """map ground truth label"""
    scenario = record.get("scenario", "")
    if scenario in _LABEL_MAP:
        return _LABEL_MAP[scenario]
    #baseline use category_id
    cat = record.get("category_id", "")
    if cat in _LABEL_MAP:
        return _LABEL_MAP[cat]
    # to fallback to query_task_type
    task = record.get("query_task_type", "")
    if task in _LABEL_MAP:
        return _LABEL_MAP[task]
    return "aggregation"  # safe default

In [14]:
def run_test(gt_path: Path, client: object) -> None:
    print(f"testing: {gt_path.name}")
    print('=' * 50)
    
    with open(gt_path, encoding="utf-8") as f:
        records = json.load(f)
    
    #skip guardrail or oos records
    record = [r for r in records if r.get("scenario") != "guardrail" and r.get("scenario") != "oos"]
    
    correct = 0
    total = 0
    mismatches = []
    per_scenario = {"simple_lookup": [0,0], "aggregation": [0,0], "filtering_ranking": [0,0]}
    
    for rec in records:
        question = rec["question"]
        expected = normalize_expected(rec)
        
        t0 = time.monotonic()
        result = route_scenario(question, client=client)
        elapsed = time.monotonic() - t0
        
        total += 1
        per_scenario[expected][1] += 1 
        
        if result.scenario == expected:
            correct += 1
            per_scenario[expected][0] += 1
            status = "CORRECT"
        else:
            status = "WRONG"
            mismatches.append({
                "id": rec.get("id", "?"),
                "question": question,
                "expected": expected,
                "got": result.scenario,
                "confidence": result.confidence,
            })
        print(f" [{status}] {rec.get('id', '?'):>5} | {result.scenario:<20} " f"(exp: {expected:<20}) conf={result.confidence:.2f} time: {elapsed:.1f}s")
    
    #summary
    print("summary: {gt_path.name}")
    print(f"overall accuracy: {correct}/{total} = {correct/total:.1%}\n")
    for sc, (c,t) in per_scenario.items():
        if t > 0:
            print(f" {sc:<22} {c}/{t} = {c/t:.1%}")
    
    if mismatches:
        print(f"\nmismatches: ({len(mismatches)}):")
        for m in mismatches:
            print(f" [{m['id']}] expected={m['expected']}, got={m['got']} (conf={m['confidence']:.2f}) question: {m['question']}")
        

In [15]:
client = build_openai_client()
v4_path = Path("../00_data/historyCalls/04_ground_truth/ground_truth_test_set_v4.json")

if v4_path.exists():
    run_test(v4_path, client)
else:
    print(f"ground truth file not found: {v4_path}")

testing: ground_truth_test_set_v4.json
 [CORRECT]  SL01 | simple_lookup        (exp: simple_lookup       ) conf=0.99 time: 3.2s
 [CORRECT]  SL02 | simple_lookup        (exp: simple_lookup       ) conf=0.98 time: 1.8s
 [CORRECT]  SL03 | simple_lookup        (exp: simple_lookup       ) conf=0.98 time: 1.8s
 [CORRECT]  SL04 | simple_lookup        (exp: simple_lookup       ) conf=0.98 time: 1.8s
 [CORRECT]  SL05 | simple_lookup        (exp: simple_lookup       ) conf=0.99 time: 1.9s
 [CORRECT]  SL06 | simple_lookup        (exp: simple_lookup       ) conf=0.97 time: 15.5s
 [CORRECT]  SL07 | simple_lookup        (exp: simple_lookup       ) conf=0.98 time: 15.4s
 [CORRECT]  SL08 | simple_lookup        (exp: simple_lookup       ) conf=0.98 time: 16.0s
 [CORRECT]  SL09 | simple_lookup        (exp: simple_lookup       ) conf=0.93 time: 16.7s
 [CORRECT]  SL10 | simple_lookup        (exp: simple_lookup       ) conf=0.96 time: 16.4s
 [CORRECT]  AG01 | aggregation          (exp: aggregation         

In [ ]:
# test baseline
baseline_path = Path("../00_data/historyCalls/04_ground_truth/ground_truth_answers_baseline_clean.json")
if baseline_path.exists():
    run_test(baseline_path, client)
else:
    print(f"ground truth file not found: {baseline_path}")